# WikiArt Inpainting - Demo End-to-End
Ten notebook pokazuje: generację maski, embeddingi, klasteryzację, inpainting i super-resolution.


In [ ]:
from pathlib import Path
import numpy as np
import torch
from datasets import load_dataset
from src.damage_generator.masks import generate_square_mask, generate_irregular_mask, apply_mask
from src.encoder.autoencoder import ConvAutoencoder, AutoencoderConfig
from src.clustering.clusterizer import cluster_embeddings
from src.inpainting.model import SimpleUNet, InpaintConfig
from src.superres.model import SimpleSRCNN, SuperResConfig
from torchvision import transforms
import matplotlib.pyplot as plt


## 1. Pobranie próbki danych


In [ ]:
dataset = load_dataset('Artificio/WikiArt_Full', split='train[:5]')
image = dataset[0]['image']
image_size = 256
transform = transforms.Compose([transforms.Resize((image_size, image_size)), transforms.ToTensor()])
image_tensor = transform(image)
plt.imshow(image_tensor.permute(1,2,0))
plt.axis('off')


## 2. Generacja maski (square / irregular)


In [ ]:
mask_square = generate_square_mask(image_size)
mask_irregular = generate_irregular_mask(image_size)
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(mask_square, cmap='gray'); ax[0].set_title('Square')
ax[1].imshow(mask_irregular, cmap='gray'); ax[1].set_title('Irregular')
for a in ax: a.axis('off')


## 3. Autoenkoder i embeddingi


In [ ]:
autoencoder = ConvAutoencoder(AutoencoderConfig())
with torch.no_grad():
    recon, z = autoencoder(image_tensor.unsqueeze(0))
z.shape


## 4. Klasteryzacja embeddingów


In [ ]:
embeddings = np.random.randn(20, 256)
labels, artifacts = cluster_embeddings(embeddings, method='kmeans', n_clusters=4)
labels[:5]


## 5. Inpainting (przykładowy model)


In [ ]:
mask = generate_square_mask(image_size)
masked = apply_mask(image_tensor.permute(1,2,0).numpy(), mask)
masked_tensor = torch.from_numpy(masked).permute(2,0,1).unsqueeze(0)
mask_tensor = torch.from_numpy(mask).unsqueeze(0).unsqueeze(0)
inpaint_model = SimpleUNet(InpaintConfig())
with torch.no_grad():
    output = inpaint_model(masked_tensor, mask_tensor)
plt.imshow(output.squeeze(0).permute(1,2,0))
plt.axis('off')


## 6. Super-resolution


In [ ]:
sr_model = SimpleSRCNN(SuperResConfig())
with torch.no_grad():
    sr = sr_model(image_tensor.unsqueeze(0))
plt.imshow(sr.squeeze(0).permute(1,2,0))
plt.axis('off')
